# Dynamic Models of Buildings and HVAC Controllers

Zone Model from Candanedo et. al 2014

<img src="images/Candanedo_RC_zone model_2014.png" width="50%">

\begin{align}
C_1 \frac{dT_1(t)}{dt} &= \frac{T_{\text{ext}}(t) - T_1(t)}{R_{1,\text{ext}}} + \frac{T_2(t) - T_1(t)}{R_{1,2}} \\[10pt]
C_2 \frac{dT_2(t)}{dt} &= \frac{T_1(t) - T_2(t)}{R_{1,2}} + \frac{T_3(t) - T_2(t)}{R_{2,3}} + 0.7 \, q_{\text{SG}}(t) \\[10pt]
C_3 \frac{dT_3(t)}{dt} &= \frac{T_2(t) - T_3(t)}{R_{2,3}} + \frac{T_{\text{ext}}(t) - T_3(t)}{R_{3,\text{ext}}} + 0.3 \, q_{\text{SG}}(t) + q_{\text{IG}}(t) + q_{\text{hc}}(t)
\end{align}

where

- $C_1, C_2, C_3$: Thermal capacitances [kJ/K]
- $R_{1,\text{ext}}, R_{1,2}, R_{2,3}, R_{3,\text{ext}}$: Thermal resistances [K/kW]
- $T_1(t), T_2(t), T_3(t)$: Node temperatures [°C]
- $T_{\text{ext}}(t)$: External/outdoor temperature [°C]
- $q_{\text{SG}}(t)$: Solar gains [kW]
- $q_{\text{IG}}(t)$: Internal gains [kW]
- $q_{\text{hc}}(t)$: Heating/cooling power [kW]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import casadi as cas

from scipy.integrate import solve_ivp
from cas_models.continuous_time.models import StateSpaceModelCT
from cas_models.discrete_time.models import StateSpaceModelDTFromCT
from cas_models.discrete_time.simulate import (
    make_n_step_simulation_function_from_model,
)
from cas_models.param_utils import make_symbolic_vars_from_kwargs


# Define parameters (example values from the paper)
C1 = 99845.92  # kJ/K
C2 = 5751.61  # kJ/K
C3 = 1000.00  # kJ/K
R1_ext = 5.00  # K/kW
R1_2 = 50.00  # K/kW
R2_3 = 1.72  # K/kW
R3_ext = 8.16  # K/kW
split_SG = 0.65  # Fraction of q_SG entering node 2 (remainder 1 - split_SG enters node 3)

# Create symbolic variables
t = cas.SX.sym("t")
x = cas.SX.sym("x", 3)  # states: [T1, T2, T3]
u = cas.SX.sym("u", 4)  # inputs: [T_ext, q_SG, q_IG, q_hc]

# Extract states
T1 = x[0]
T2 = x[1]
T3 = x[2]

# Extract inputs
T_ext = u[0]
q_SG = u[1]
q_IG = u[2]
q_hc = u[3]

# Define ODEs (right-hand side)
dT1_dt = (1 / C1) * ((T_ext - T1) / R1_ext + (T2 - T1) / R1_2)
dT2_dt = (1 / C2) * ((T1 - T2) / R1_2 + (T3 - T2) / R2_3 + split_SG * q_SG)
dT3_dt = (1 / C3) * (
    (T2 - T3) / R2_3
    + (T_ext - T3) / R3_ext
    + (1 - split_SG) * q_SG
    + q_IG
    + q_hc
)

# Combine into state derivative vector
dx_dt = cas.vertcat(dT1_dt, dT2_dt, dT3_dt)

# Create CasADi function
f_casadi = cas.Function("f", [t, x, u], [dx_dt], ["t", "x", "u"], ["dx_dt"])


# Define input profiles as functions of time
def get_inputs(t):
    """Generate time-varying inputs"""
    t_hours = t / 3600  # Convert seconds to hours

    # External temperature: constant for 2 hrs, then ramp down
    if t_hours < 2.0:
        T_ext = 10.0  # °C
    else:
        T_ext = 10.0 - 2.0 * (t_hours - 2.0)  # Ramp down at 2°C/hr

    # Solar gains: step change at 1.5 hours
    q_SG = 0.5 if t_hours < 1.5 else 2.0  # kW

    # Internal gains: step change at 3 hours
    q_IG = 0.3 if t_hours < 3.0 else 1.5  # kW

    # Heating/cooling: step change at 4.5 hours
    q_hc = 0.0 if t_hours < 4.5 else 3.0  # kW

    return np.array([T_ext, q_SG, q_IG, q_hc])


# Wrapper function for scipy's solve_ivp
def ode_rhs(t, x_vec):
    """Right-hand side function for ODE solver"""
    u_vec = get_inputs(t)
    dx_dt = f_casadi(t, x_vec, u_vec)
    return np.array(dx_dt).flatten()


# Simulation parameters
t_start = 0.0  # seconds
t_end = 6 * 3600  # 6 hours in seconds
x0 = np.array([20.0, 20.0, 20.0])  # Initial temperatures [°C]

# Solve ODE
sol = solve_ivp(
    ode_rhs,
    [t_start, t_end],
    x0,
    method="RK45",
    dense_output=True,
    max_step=60,
)  # max step of 60 seconds

# Create dense time vector for plotting
t_plot = np.linspace(t_start, t_end, 500)
x_plot = sol.sol(t_plot)

# Get input profiles for plotting
inputs_plot = np.array([get_inputs(t) for t in t_plot]).T

# Convert time to hours for plotting
t_hours = t_plot / 3600

# Create plots
fig, axes = plt.subplots(2, 1, figsize=(7, 5.5))

# Plot temperatures
axes[0].plot(t_hours, x_plot[0, :], label="$T_1$ (Node 1)")
axes[0].plot(t_hours, x_plot[1, :], label="$T_2$ (Node 2)")
axes[0].plot(t_hours, x_plot[2, :], label="$T_3$ (Zone Temperature)")
axes[0].plot(
    t_hours,
    inputs_plot[0, :],
    "--",
    label="$T_{ext}$ (External)",
    linewidth=2,
    alpha=0.7,
)
axes[0].set_xlabel("Time [hours]")
axes[0].set_ylabel("Temperature [°C]")
axes[0].set_title("Zone Thermal Response")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot inputs
axes[1].plot(t_hours, inputs_plot[1, :], label="$q_{SG}$ (Solar gains)")
axes[1].plot(t_hours, inputs_plot[2, :], label="$q_{IG}$ (Internal gains)")
axes[1].plot(t_hours, inputs_plot[3, :], label="$q_{hc}$ (Heating/cooling)")
axes[1].set_xlabel("Time [hours]")
axes[1].set_ylabel("Heat Flow [kW]")
axes[1].set_title("Input Disturbances and Control")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final states
print("\nFinal temperatures after 6 hours:")
print(f"T1 = {x_plot[0, -1]:.2f} °C")
print(f"T2 = {x_plot[1, -1]:.2f} °C")
print(f"T3 (Zone) = {x_plot[2, -1]:.2f} °C")

# x_plot[:, -1]  # plot actual values
assert np.allclose(x_plot[:, -1], [19.46821058, 24.4205289, 27.35597101])

In [ ]:
def make_zone_rcmodel(
    C1=None,
    C2=None,
    C3=None,
    R1_ext=None,
    R1_2=None,
    R2_3=None,
    R3_ext=None,
    split_SG=None,
):

    # Symbolic time, states, and inputs
    t = cas.SX.sym("t")
    x = cas.SX.sym("x", 3)  # states: [T1, T2, T3]
    u = cas.SX.sym("u", 4)  # inputs: [T_ext, q_SG, q_IG, q_hc]

    T1, T2, T3 = x[0], x[1], x[2]
    T_ext, q_SG, q_IG, q_hc = u[0], u[1], u[2], u[3]

    # Parameters (RC values + solar-gain split fraction). Any left as
    # None become symbolic CasADi variables; any given a numeric value
    # are substituted directly.
    params = make_symbolic_vars_from_kwargs(
        C1=C1,
        C2=C2,
        C3=C3,
        R1_ext=R1_ext,
        R1_2=R1_2,
        R2_3=R2_3,
        R3_ext=R3_ext,
        split_SG=split_SG,
    )
    C1 = params["C1"]
    C2 = params["C2"]
    C3 = params["C3"]
    R1_ext = params["R1_ext"]
    R1_2 = params["R1_2"]
    R2_3 = params["R2_3"]
    R3_ext = params["R3_ext"]
    split_SG = params["split_SG"]

    # ODEs (right-hand side)
    dT1_dt = (1 / C1) * ((T_ext - T1) / R1_ext + (T2 - T1) / R1_2)
    dT2_dt = (1 / C2) * ((T1 - T2) / R1_2 + (T3 - T2) / R2_3 + split_SG * q_SG)
    dT3_dt = (1 / C3) * (
        (T2 - T3) / R2_3
        + (T_ext - T3) / R3_ext
        + (1 - split_SG) * q_SG
        + q_IG
        + q_hc
    )
    rhs = cas.vertcat(dT1_dt, dT2_dt, dT3_dt)
    y = T3

    # Keep only the parameters that are still symbolic (i.e. left as
    # None above) — numeric ones are already baked into rhs/y and must
    # NOT be listed as Function arguments (CasADi requires every
    # Function argument to be purely symbolic).
    symbolic_params = {
        name: value
        for name, value in params.items()
        if isinstance(value, cas.SX)
    }

    # State transition function — f(t, x, u, <symbolic params...>) -> rhs
    f = cas.Function(
        "f",
        [t, x, u, *symbolic_params.values()],
        [rhs],
        ["t", "x", "u", *symbolic_params.keys()],
        ["rhs"],
    )

    # Output function — single output, zone temperature T3
    h = cas.Function(
        "h",
        [t, x, u, *symbolic_params.values()],
        [y],
        ["t", "x", "u", *symbolic_params.keys()],
        ["y"],
    )

    # Assemble the continuous-time state-space model
    model = StateSpaceModelCT(
        f,
        h,
        n=3,
        nu=4,
        ny=1,
        params=symbolic_params,
        name="zone_model",
        input_names=["T_ext", "q_SG", "q_IG", "q_hc"],
        state_names=["T1", "T2", "T3"],
        output_names=["T3"],
    )

    return model


params = dict(
    C1=99845.92,
    C2=5751.61,
    C3=1000.00,
    R1_ext=5.00,
    R1_2=50.00,
    R2_3=1.72,
    R3_ext=8.16,
    split_SG=0.65,
)

model_ct = make_zone_rcmodel(**params)
model_ct.describe()

In [ ]:
# --- Discretization setup (all time quantities in seconds, matching
#     the continuous-time model's implicit units: R in K/kW, C in kJ/K) ---
dt = 60.0  # time step [s] (i.e. 1-minute intervals)
t_end = 6 * 3600.0  # simulation horizon [s] (6 hours)
nT = int(t_end / dt)  # number of steps (360)

model_dt = StateSpaceModelDTFromCT(model_ct, dt)
simulate = make_n_step_simulation_function_from_model(model_dt, nT=nT)


# --- Input profiles (unchanged; already defined in seconds) ---
def get_inputs(t):
    """Generate time-varying inputs. t is in seconds."""
    t_hours = t / 3600  # Convert seconds to hours

    if t_hours < 2.0:
        T_ext = 10.0
    else:
        T_ext = 10.0 - 2.0 * (t_hours - 2.0)

    q_SG = 0.5 if t_hours < 1.5 else 2.0
    q_IG = 0.3 if t_hours < 3.0 else 1.5
    q_hc = 0.0 if t_hours < 4.5 else 3.0

    return np.array([T_ext, q_SG, q_IG, q_hc])


# --- Simulation ---
x0 = np.array([20.0, 20.0, 20.0])
t_eval = np.linspace(0.0, t_end, nT + 1)  # length nT+1
U = np.array(
    [get_inputs(tk) for tk in t_eval[:-1]]
)  # shape (nT, 4), zero-order hold

X, Y = simulate(t_eval, U, x0)
X = np.array(X)  # shape (nT+1, 3): [T1, T2, T3] per row
Y = np.array(Y)  # shape (nT+1, 1): T3 (zone output)

# --- Plotting ---
inputs_plot = np.array([get_inputs(tk) for tk in t_eval]).T
t_hours = t_eval / 3600

fig, axes = plt.subplots(2, 1, figsize=(7, 5.5))

axes[0].plot(t_hours, X[:, 0], label="$T_1$ (Node 1)")
axes[0].plot(t_hours, X[:, 1], label="$T_2$ (Node 2)")
axes[0].plot(t_hours, X[:, 2], label="$T_3$ (Zone Temperature)")
axes[0].plot(
    t_hours,
    inputs_plot[0, :],
    "--",
    label="$T_{ext}$ (External)",
    linewidth=2,
    alpha=0.7,
)
axes[0].set_xlabel("Time [hours]")
axes[0].set_ylabel("Temperature [°C]")
axes[0].set_title("Zone Thermal Response (Discrete-time CasADi simulation)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_hours, inputs_plot[1, :], label="$q_{SG}$ (Solar gains)")
axes[1].plot(t_hours, inputs_plot[2, :], label="$q_{IG}$ (Internal gains)")
axes[1].plot(t_hours, inputs_plot[3, :], label="$q_{hc}$ (Heating/cooling)")
axes[1].set_xlabel("Time [hours]")
axes[1].set_ylabel("Heat Flow [kW]")
axes[1].set_title("Input Disturbances and Control")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Final states ---
print("\nFinal temperatures after 6 hours:")
print(f"T1 = {X[-1, 0]:.2f} °C")
print(f"T2 = {X[-1, 1]:.2f} °C")
print(f"T3 (Zone) = {X[-1, 2]:.2f} °C")

assert np.allclose(X[-1, :], [19.46872206, 24.4365923, 27.37296162])